<a href="https://colab.research.google.com/github/cbonnin88/EcoVolt-HR/blob/main/data_generation_EcoVolt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.1 MB/s eta 0:00:00


In [38]:
import polars as pl
import numpy as np
from faker import Faker
from datetime import date, timedelta

In [39]:
locales = {
    'Germany':'de_DE',
    'France':'fr_Fr',
    'United Kingdom':'en_GB',
    'Norway':'no_NO',
    'Denmark':'da_DK',
    'Spain':'es_ES',
    'The Netherlands':'nl_NL'
}

In [54]:
fakers = {country:Faker(locale) for country, locale in locales.items()}

In [53]:
num_total = 12000
num_leadership = 6
num_staff = num_total - num_leadership
today = date(2026,1,1)

In [55]:
dept_titles = {
    'Wind Operations': ['Turbine Technician', 'Wind Farm Manager', 'Maintenance Engineer'],
    'Solar Engineering': ['PV Designer', 'Solar Project Engineer', 'Grid Connection Specialist'],
    'Green Hydrogen R&D': ['Electrolysis Researcher', 'Chemical Engineer', 'Hydrogen Storage Expert'],
    'Grid Infrastructure': ['High Voltage Engineer', 'Grid Analyst', 'Substation Manager'],
    'Corporate': ['HR Business Partner', 'Financial Analyst', 'Legal Counsel', 'IT Support']
}

In [56]:
leadership_titles = ['CEO', 'CFO', 'Chief People Officer', 'CTO', 'Chief of Operations', 'Chief of Research']
countries_list = list(locales.keys())

In [57]:
np.random.seed(42)
staff_records = []

In [58]:
for i in range(1,num_staff + 1):
  country = np.random.choice(countries_list)
  dept = np.random.choice(list(dept_titles.keys()))
  fake = fakers[country]
  hire_date = date(2020,1,1) + timedelta(days=np.random.randint(0,3800))

  staff_records.append({
      'Employee_ID': f'EV{i:05d}',
      'first_name': fake.first_name(),
      'last_name': fake.last_name(),
      'Country': country,
      'Department': dept,
      'Job_Title': np.random.choice(dept_titles[dept]),
      'Job_Level': np.random.choice(['T2','T3','T4','T5'],p=[0.05,0.20,0.40,0.35]),
      'Remote_Type': np.random.choice(['Hybrid','Office','Remote'], p=[0.25,0.15,0.60]),
      'Hire_Date': hire_date
  })

In [59]:
leadership_records = []
for i, title in enumerate(leadership_titles):
  country = np.random.choice(countries_list)
  fake = fakers[country]
  hire_date = date(2020,1,1) + timedelta(days=np.random.randint(0,1000))

  leadership_records.append({
      'Employee_ID': f'EV{num_staff + i + 1:05d}',
      'first_name': fake.first_name(),
      'last_name': fake.last_name(),
      'Country': country,
      'Department': 'Leadership',
      'Remote_Type': np.random.choice(['Hybrid','Office','Remote'], p=[0.25,0.15,0.60]),
      'Job_Title': title,
      'Job_Level': 'T1',
      'Hire_Date': hire_date
  })

In [60]:
employees = pl.DataFrame(staff_records + leadership_records)
employees = employees.with_columns(
    ((pl.lit(today) - pl.col('Hire_Date')).dt.total_days() / 365.25).round(1).alias('Tenure_Years')
)

In [61]:
midpoints = {'T1':240000,'T2':145000,'T3':95000,'T4':70000,'T5':48000}

In [62]:
comp_records = []
for row in employees.iter_rows(named=True):
  level = row['Job_Level']
  mid = midpoints[level]
  # Adding noise: Tenure slightly increases pay, but performance rating has a bigger impact
  tenure_factor = 1 + (row['Tenure_Years'] * 0.01)
  perf_factor = np.random.uniform(0.85,1.15)
  base_salary = mid * tenure_factor * perf_factor

  comp_records.append({
    'Employee_ID': row['Employee_ID'],
    'Annual_Base_EUR': round(base_salary,2),
    'Market_Midpoint_EUR': mid,
    'Bonus_Target_Pct': 0.35 if level == 'T1' else np.random.choice([0.05,0.1,0.15,0.2]),
    'Performance_Rating': np.random.choice([1,2,3,4,5], p= [0.05,0.15,0.5,0.2,0.1])
})

In [63]:
compensation = pl.DataFrame(comp_records)

In [64]:
display(employees.head())

Employee_ID,first_name,last_name,Country,Department,Job_Title,Job_Level,Remote_Type,Hire_Date,Tenure_Years
str,str,str,str,str,str,str,str,date,f64
"""EV00001""","""Lois""","""de Vries""","""The Netherlands""","""Grid Infrastructure""","""Substation Manager""","""T5""","""Remote""",2022-05-10,3.6
"""EV00002""","""Marijn""","""van Rossum""","""The Netherlands""","""Solar Engineering""","""Grid Connection Specialist""","""T3""","""Remote""",2021-04-11,4.7
"""EV00003""","""Anne""","""Vik""","""Norway""","""Green Hydrogen R&D""","""Electrolysis Researcher""","""T5""","""Remote""",2024-08-12,1.4
"""EV00004""","""Graciana""","""Ibáñez""","""Spain""","""Solar Engineering""","""PV Designer""","""T4""","""Remote""",2023-04-30,2.7
"""EV00005""","""Ofelia""","""Toft""","""Denmark""","""Grid Infrastructure""","""High Voltage Engineer""","""T4""","""Office""",2027-12-14,-1.9


In [65]:
display(compensation.head())

Employee_ID,Annual_Base_EUR,Market_Midpoint_EUR,Bonus_Target_Pct,Performance_Rating
str,f64,i64,f64,i64
"""EV00001""",51296.74,48000,0.1,3
"""EV00002""",110362.49,95000,0.1,4
"""EV00003""",42517.02,48000,0.15,3
"""EV00004""",78837.76,70000,0.15,3
"""EV00005""",70426.94,70000,0.1,3


In [66]:
employees.write_csv('employees.csv')
compensation.write_csv('compensation.csv')

In [68]:
print('--- Data Creation Complete ---')
print(f'Total Employees: {len(employees)}')
display(employees.select(['first_name','last_name','Country','Job_Title','Tenure_Years','Remote_Type']).head(5))

--- Data Creation Complete ---
Total Employees: 12000


first_name,last_name,Country,Job_Title,Tenure_Years,Remote_Type
str,str,str,str,f64,str
"""Lois""","""de Vries""","""The Netherlands""","""Substation Manager""",3.6,"""Remote"""
"""Marijn""","""van Rossum""","""The Netherlands""","""Grid Connection Specialist""",4.7,"""Remote"""
"""Anne""","""Vik""","""Norway""","""Electrolysis Researcher""",1.4,"""Remote"""
"""Graciana""","""Ibáñez""","""Spain""","""PV Designer""",2.7,"""Remote"""
"""Ofelia""","""Toft""","""Denmark""","""High Voltage Engineer""",-1.9,"""Office"""
